# Turbo Code Tutorial 

## Imports and setup

In [ ]:
import sys
from pathlib import Path
import importlib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent, cwd.parent.parent]

repo_root = None
for c in candidates:
    if (c / "turbo" / "__init__.py").exists():
        repo_root = c
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not find the repository root. Put this notebook inside the repo, "
        "for example in tutorials/."
    )

repo_root_str = str(repo_root)
if repo_root_str in sys.path:
    sys.path.remove(repo_root_str)
sys.path.insert(0, repo_root_str)

for name in list(sys.modules):
    if name == "turbo" or name.startswith("turbo."):
        del sys.modules[name]

importlib.invalidate_caches()

import turbo
import turbo.config as config
import turbo.encoder as encoder
import turbo.decoder as decoder

print("Repository root:", repo_root)
print("Turbo package:", turbo.__file__)
print("Encoder module:", encoder.__file__)

## Tutorial parameters

The original project is configured for larger experiments.  
For a tutorial, we keep the setup intentionally small and fast.


In [ ]:
TUTORIAL_BITS = 128
TUTORIAL_ITERATIONS = [1, 2, 3]
TUTORIAL_TURBO_EBN0_DB = np.array([0.0, 0.5, 1.0, 1.5, 2.0])
TUTORIAL_CONV_EBN0_DB = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
TUTORIAL_RATE_LABEL = "1/3"
SEED = config.RANDOM_SEED

settings_df = pd.DataFrame({
    "Parameter": [
        "Information bits",
        "Iterations",
        "Turbo Eb/N0 points",
        "Conv Eb/N0 points",
        "Rate label",
        "Seed",
    ],
    "Value": [
        TUTORIAL_BITS,
        str(TUTORIAL_ITERATIONS),
        str(TUTORIAL_TURBO_EBN0_DB.tolist()),
        str(TUTORIAL_CONV_EBN0_DB.tolist()),
        TUTORIAL_RATE_LABEL,
        SEED,
    ],
})
settings_df

## Helper functions

These notebook-level helper functions keep the tutorial robust and fast while still using the actual project encoder and decoder.


In [ ]:
def tutorial_interleaver(seed: int, information_bits: int):
    rng = np.random.default_rng(seed)
    p = rng.permutation(information_bits).astype(int)
    return p, np.argsort(p).astype(int)

def tutorial_turbo_encode(info_bits, interleaver, rate_label):
    info_bits = np.asarray(info_bits, dtype=np.int8)
    systematic_stream, parity_stream_1 = encoder.encode_rsc_terminated(info_bits)
    _, parity_stream_2 = encoder.encode_rsc_terminated(info_bits[interleaver])

    total_len = len(systematic_stream)
    keep_pattern_1, keep_pattern_2 = config.get_puncture_definition(rate_label)

    keep_mask_1 = np.ones(total_len, dtype=np.int8)
    keep_mask_2 = np.ones(total_len, dtype=np.int8)
    keep_mask_1[:len(info_bits)] = np.resize(keep_pattern_1, len(info_bits))
    keep_mask_2[:len(info_bits)] = np.resize(keep_pattern_2, len(info_bits))

    return {
        "systematic_stream_1": systematic_stream,
        "parity_stream_1_full": parity_stream_1,
        "parity_stream_2_full": parity_stream_2,
        "transmitted_parity_stream_1": parity_stream_1[keep_mask_1 == 1],
        "transmitted_parity_stream_2": parity_stream_2[keep_mask_2 == 1],
        "parity_keep_mask_1": keep_mask_1,
        "parity_keep_mask_2": keep_mask_2,
    }

def tutorial_depuncture(received, keep_mask):
    full = np.zeros(len(keep_mask), dtype=float)
    tx_index = 0
    for index in range(len(keep_mask)):
        if keep_mask[index] == 1:
            full[index] = received[tx_index]
            tx_index += 1
    return full

def llr_to_bits(llr):
    return (llr < 0.0).astype(np.int8)

def plot_convergence(history, iterations, n_bits=20):
    fig, ax = plt.subplots(figsize=(9, 4.8))
    x = np.arange(n_bits)
    markers = ["o", "s", "^", "D", "v", "P"]
    for idx, it in enumerate(iterations):
        ax.plot(
            x,
            history[idx, :n_bits],
            marker=markers[idx % len(markers)],
            linewidth=1.8,
            markersize=5,
            label=f"Iteration {it}",
        )
    ax.axhline(0.0, linewidth=1.0, linestyle="--")
    ax.set_xlabel("Bit index")
    ax.set_ylabel("LLR")
    ax.set_title("Turbo decoder convergence on one block")
    ax.grid(True, linestyle="--", alpha=0.35)
    ax.legend(frameon=True)
    fig.tight_layout()
    return fig, ax

def plot_tiny_turbo_ber(ebn0, turbo_results, iterations):
    fig, ax = plt.subplots(figsize=(8, 4.8))
    markers = ["o", "s", "^", "D", "v", "P"]
    for idx, it in enumerate(iterations):
        ax.semilogy(
            ebn0,
            np.clip(turbo_results[it], 1e-8, None),
            marker=markers[idx % len(markers)],
            linewidth=2.0,
            markersize=6,
            label=f"Iteration {it}",
        )
    ax.set_xlabel("Eb/N0 (dB)")
    ax.set_ylabel("BER")
    ax.set_title("Illustrative turbo BER curves")
    ax.grid(True, which="both", linestyle="--", alpha=0.35)
    ax.legend(frameon=True)
    fig.tight_layout()
    return fig, ax

def plot_tiny_conv_ber(ebn0, uncoded, coded):
    fig, ax = plt.subplots(figsize=(8, 4.8))
    ax.semilogy(ebn0, np.clip(uncoded, 1e-8, None), marker="o", linewidth=2.0, markersize=6, label="Uncoded BPSK")
    ax.semilogy(ebn0, np.clip(coded, 1e-8, None), marker="s", linewidth=2.0, markersize=6, label="Conv. coded + soft Viterbi")
    ax.set_xlabel("Eb/N0 (dB)")
    ax.set_ylabel("BER")
    ax.set_title("Illustrative convolutional baseline")
    ax.grid(True, which="both", linestyle="--", alpha=0.35)
    ax.legend(frameon=True)
    fig.tight_layout()
    return fig, ax

def illustrative_turbo_results():
    ebn0 = TUTORIAL_TURBO_EBN0_DB
    results = {
        1: np.array([1.8e-1, 1.2e-1, 7.2e-2, 3.2e-2, 1.4e-2]),
        2: np.array([1.4e-1, 8.0e-2, 3.8e-2, 1.2e-2, 3.8e-3]),
        3: np.array([1.2e-1, 6.5e-2, 2.6e-2, 6.5e-3, 1.6e-3]),
    }
    return ebn0, results

def illustrative_conv_results():
    ebn0 = TUTORIAL_CONV_EBN0_DB
    uncoded = np.array([1.30e-1, 1.05e-1, 7.9e-2, 5.7e-2, 3.8e-2])
    coded = np.array([9.5e-2, 6.5e-2, 3.4e-2, 1.15e-2, 2.9e-3])
    return ebn0, uncoded, coded

print("Helper functions defined.")

## Interleaver

A turbo code uses two constituent encoders.  
The second encoder processes an interleaved version of the original information sequence:

$$
\mathbf{u}_{\pi} = (u_{\pi(1)}, u_{\pi(2)}, \dots, u_{\pi(K)})
$$


In [ ]:
interleaver, deinterleaver = tutorial_interleaver(SEED, TUTORIAL_BITS)

inter_df = pd.DataFrame({
    "Position": np.arange(20),
    "Interleaver index": interleaver[:20],
    "Inverse map": deinterleaver[interleaver[:20]],
})
inter_df

## Turbo encoding of one short block

Before puncturing, the classical turbo code structure is approximately

$$
\mathbf{c} = \left(\mathbf{u}, \mathbf{p}^{(1)}, \mathbf{p}^{(2)}\right)
$$


In [ ]:
rng = np.random.default_rng(SEED)
bits = rng.integers(0, 2, TUTORIAL_BITS, dtype=np.int8)
encoded = tutorial_turbo_encode(bits, interleaver, TUTORIAL_RATE_LABEL)

summary_df = pd.DataFrame({
    "Stream": [
        "Information bits",
        "Systematic stream",
        "Parity stream 1 (full)",
        "Parity stream 2 (full)",
        "Transmitted parity 1",
        "Transmitted parity 2",
    ],
    "Length": [
        len(bits),
        len(encoded["systematic_stream_1"]),
        len(encoded["parity_stream_1_full"]),
        len(encoded["parity_stream_2_full"]),
        len(encoded["transmitted_parity_stream_1"]),
        len(encoded["transmitted_parity_stream_2"]),
    ],
})
summary_df

In [ ]:
preview_df = pd.DataFrame({
    "u[k]": bits[:20],
    "systematic": encoded["systematic_stream_1"][:20],
    "parity_1": encoded["parity_stream_1_full"][:20],
    "parity_2": encoded["parity_stream_2_full"][:20],
    "keep_1": encoded["parity_keep_mask_1"][:20],
    "keep_2": encoded["parity_keep_mask_2"][:20],
})
preview_df

## Effective code rate

With puncturing, the effective rate becomes

$$
R_{\mathrm{eff}} = \frac{K}{N_{\mathrm{tx}}}
$$


In [ ]:
total_len = len(encoded["systematic_stream_1"])
tx_count = total_len + int(np.sum(encoded["parity_keep_mask_1"])) + int(np.sum(encoded["parity_keep_mask_2"]))
effective_rate = TUTORIAL_BITS / tx_count

rate_df = pd.DataFrame({
    "Quantity": ["Information bits K", "Systematic length", "Total transmitted symbols", "Effective rate"],
    "Value": [TUTORIAL_BITS, total_len, tx_count, round(float(effective_rate), 6)],
})
rate_df

## BPSK transmission through AWGN

Each transmitted coded bit is mapped to BPSK using

$$
x = 1 - 2c
$$

and the received sample is

$$
y = x + n, \qquad n \sim \mathcal{N}(0, \sigma^2)
$$


In [ ]:
ebn0_db = 0.8
sigma2 = config.sigma2_from_ebn0(ebn0_db, effective_rate)
sigma = np.sqrt(sigma2)

tx_sys = 1.0 - 2.0 * encoded["systematic_stream_1"]
tx_p1 = 1.0 - 2.0 * encoded["transmitted_parity_stream_1"]
tx_p2 = 1.0 - 2.0 * encoded["transmitted_parity_stream_2"]

rx_sys = tx_sys + sigma * rng.standard_normal(len(tx_sys))
rx_p1 = tx_p1 + sigma * rng.standard_normal(len(tx_p1))
rx_p2 = tx_p2 + sigma * rng.standard_normal(len(tx_p2))

channel_df = pd.DataFrame({
    "tx_sys": np.round(tx_sys[:12], 3),
    "rx_sys": np.round(rx_sys[:12], 3),
})
channel_df

## Depuncturing

The decoder expects full-length parity reliability streams.  
Missing punctured positions are filled with zero reliability before decoding.


In [ ]:
rx_p1_full = tutorial_depuncture(rx_p1, encoded["parity_keep_mask_1"])
rx_p2_full = tutorial_depuncture(rx_p2, encoded["parity_keep_mask_2"])

depuncture_df = pd.DataFrame({
    "Index": np.arange(15),
    "Keep mask 1": encoded["parity_keep_mask_1"][:15],
    "Depunctured parity 1": np.round(rx_p1_full[:15], 3),
    "Keep mask 2": encoded["parity_keep_mask_2"][:15],
    "Depunctured parity 2": np.round(rx_p2_full[:15], 3),
})
depuncture_df

## One iterative turbo-decoding example

The decoder runs iterative Max-Log-MAP processing and exchanges extrinsic information between the two constituent decoders.

The posterior LLR is conceptually

$$
L^{\mathrm{post}}(u_k) = L^{\mathrm{sys}}(u_k) + L^{\mathrm{apr}}(u_k) + L^{\mathrm{ext}}(u_k)
$$


In [ ]:
history = decoder.decode_turbo(
    received_systematic_stream_1=rx_sys,
    received_parity_stream_1_full=rx_p1_full,
    received_parity_stream_2_full=rx_p2_full,
    sigma2=sigma2,
    iteration_count=max(TUTORIAL_ITERATIONS),
    interleaver=interleaver,
    information_bits=TUTORIAL_BITS,
)

llr_df = pd.DataFrame({
    "Bit index": np.arange(12),
    "Iteration 1": np.round(history[0, :12], 3),
    "Iteration 2": np.round(history[1, :12], 3),
    "Iteration 3": np.round(history[2, :12], 3),
})
llr_df

In [ ]:
error_rows = []
for idx, it in enumerate(TUTORIAL_ITERATIONS):
    hard = llr_to_bits(history[idx])
    errors = int(np.sum(bits != hard))
    error_rows.append({"Iteration": it, "Bit errors on this block": errors})
pd.DataFrame(error_rows)

In [ ]:
plot_convergence(history, TUTORIAL_ITERATIONS, n_bits=20)
plt.show()

### Result explanation: decoder convergence on one block

This figure shows the **soft-output LLRs** for the first few bits after each turbo-decoder iteration.

How to read it:

- values **far from zero** mean the decoder is more confident about the bit decision
- values **close to zero** mean the bit is still uncertain
- if a point moves farther away from zero from iteration 1 to iteration 3, the decoder is gaining confidence in that bit

This is the expected behavior of turbo decoding. A turbo decoder exchanges **extrinsic information** between two constituent decoders, so the bit reliabilities are refined over multiple iterations. In classical turbo-code descriptions, this iterative exchange of soft information is the core reason performance improves from one iteration to the next, while the gain usually becomes smaller after the first few iterations.  


## Illustrative BER curves for turbo decoding

For a tutorial notebook, very small Monte Carlo runs often produce jagged and unconvincing BER curves.  
The table and plot below therefore use **representative smooth values** that match the expected qualitative behavior of turbo codes:

- BER falls as $E_b/N_0$ increases
- later iterations outperform earlier ones
- the gain from iteration 2 to 3 is smaller than the gain from iteration 1 to 2


In [ ]:
ebn0_turbo, turbo_results = illustrative_turbo_results()

ber_df = pd.DataFrame({"Eb/N0 (dB)": ebn0_turbo})
for it in TUTORIAL_ITERATIONS:
    ber_df[f"BER it={it}"] = turbo_results[it]
ber_df

In [ ]:
plot_tiny_turbo_ber(ebn0_turbo, turbo_results, TUTORIAL_ITERATIONS)
plt.show()

### Result explanation: illustrative turbo BER curves

These BER curves summarize the expected **waterfall behavior** of turbo codes.

Key observations:

- as $E_b/N_0$ increases, the BER decreases
- iteration 2 performs better than iteration 1
- iteration 3 performs better than iteration 2
- the improvement from later iterations is smaller than the first improvement, showing **diminishing returns**

This matches standard turbo-code behavior: a turbo code is built from two constituent encoders separated by an interleaver, and iterative decoding improves the reliability estimates step by step. The interleaver helps reduce the chance that both constituent encoders produce weak parity protection for the same information pattern, which is one reason turbo codes achieve strong performance.  


## Illustrative convolutional baseline

This plot provides a clean baseline comparison for the tutorial:

- uncoded BPSK decreases gradually
- convolutionally coded BPSK drops faster
- turbo decoding goes further through iterative processing


In [ ]:
ebn0_conv, uncoded, coded = illustrative_conv_results()

conv_df = pd.DataFrame({
    "Eb/N0 (dB)": ebn0_conv,
    "Uncoded BER": uncoded,
    "Convolutional coded BER": coded,
})
conv_df

In [ ]:
plot_tiny_conv_ber(ebn0_conv, uncoded, coded)
plt.show()

### Result explanation: convolutional baseline

This baseline plot is useful because it shows the improvement from **classical coding** before turbo decoding is introduced.

How to interpret it:

- the **uncoded BPSK** curve decreases gradually with increasing $E_b/N_0$
- the **convolutionally coded** curve drops faster, showing coding gain from added redundancy and soft Viterbi decoding
- compared with this baseline, turbo decoding aims to go further by using **iterative soft-input soft-output decoding**

This is a good tutorial comparison because it shows the progression from uncoded transmission, to conventional trellis coding, to turbo coding with interleaving and iterative decoding.  
